# Qwen3 0.6B Fine-Tuning — Regular Causal LM Spam Detection

Fine-tunes `Qwen/Qwen3-0.6B` as a regular causal language model on the combined
TREC-2007 + CEAS-2008 spam dataset using LoRA.

Instead of attaching a classification head, the model is trained to generate a
single label: `spam` or `ham`.


In [1]:
import hashlib
import os
import re
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import torch
from aim_tracking import create_aim_callbacks, summarize_text_classification_dataset
from datasets import ClassLabel, DatasetDict, load_dataset
from dataset.combine import combine_datasets
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from trl import SFTConfig, SFTTrainer


In [2]:
MODEL_ID = "Qwen/Qwen3-0.6B"

SEED = 67
TRAIN_SPLIT = 0.95
VALIDATION_SPLIT = 0.006
TEST_SPLIT = 0.04
HOLDOUT_SPLIT = VALIDATION_SPLIT + TEST_SPLIT
MAX_SEQ_LENGTH = 512

# More conservative than the sequence-classification notebook because causal LM
# training has a larger memory footprint on the same hardware.
TRAIN_BATCH_SIZE = 24
EVAL_BATCH_SIZE = 15
GRADIENT_ACCUMULATION_STEPS = 1
NUM_TRAIN_EPOCHS = 1
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 0.5

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
MPS_MEMORY_FRACTION = 0.92

POSITIVE_LABEL_TEXT = "spam"
NEGATIVE_LABEL_TEXT = "ham"
LABEL_TEXT_TO_ID = {
    NEGATIVE_LABEL_TEXT: 0,
    POSITIVE_LABEL_TEXT: 1,
    "valid": 0,
    "not spam": 0,
    "no": 0,
    "yes": 1,
}

AIM_EXPERIMENT_NAME = "qwen3-0.6b-spam-regular-causal-lm"
AIM_SYSTEM_TRACKING_INTERVAL = 10

set_seed(SEED)


In [3]:
def find_device() -> str:
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


DEVICE = find_device()
CUDA = DEVICE == "cuda"
IS_MPS = DEVICE == "mps"

if IS_MPS:
    if hasattr(torch.mps, "empty_cache"):
        torch.mps.empty_cache()
    if hasattr(torch.mps, "set_per_process_memory_fraction"):
        torch.mps.set_per_process_memory_fraction(MPS_MEMORY_FRACTION)

print(f"Using device: {DEVICE.upper()}")


Using device: CUDA


## Dataset

Download and prepare the TREC-2007 + CEAS-2008 dataset before training.
The combined parquet is cached and reused on later runs.


In [4]:
project_root = Path.cwd().resolve()
if not (project_root / "dataset").exists():
    project_root = project_root.parent

AIM_REPO_PATH = str(project_root)
DATASET_PATH = combine_datasets(["trec_2007", "ceas_2008"], spam_ham_ratio=0.5)


def sha256_file(path: str, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


DATASET_SHA256 = sha256_file(DATASET_PATH)

dataset = load_dataset("parquet", data_files=DATASET_PATH, split="train")
dataset = dataset.cast_column("label", ClassLabel(names=["valid", "spam"]))
dataset_summary = summarize_text_classification_dataset(dataset)
dataset_label_counts = dataset_summary["label_counts"]
dataset_source_counts = dataset_summary["source_counts"]
dataset_text_stats = dataset_summary["text_stats"]

print(f"Loaded {len(dataset)} rows from {DATASET_PATH}")
print(f"Dataset SHA256: {DATASET_SHA256}")
print(f"Columns: {dataset.column_names}")
print(f"Spam: {dataset_label_counts['spam']}, Ham: {dataset_label_counts['ham']}")
print(f"Source counts: {dataset_source_counts}")
print(f"Average subject chars: {dataset_text_stats['avg_subject_chars']:.2f}")
print(f"Average body chars: {dataset_text_stats['avg_body_chars']:.2f}")


Combined dataset already exists: /home/ubuntu/masters-thesis/dataset/combined_datasets/generated/ceas_2008__trec_2007__dedupe_high__spam_0_5__0b6c8e960e.parquet
Loaded 81206 rows from /home/ubuntu/masters-thesis/dataset/combined_datasets/generated/ceas_2008__trec_2007__dedupe_high__spam_0_5__0b6c8e960e.parquet
Dataset SHA256: a521e344124a1782762a07ce862ff36a134a672f4801e4855573c8ebc93c22cb
Columns: ['subject', 'body', 'label', 'source']
Spam: 40603, Ham: 40603
Source counts: {'trec_2007': 50019, 'ceas_2008': 31187}
Average subject chars: 41.13
Average body chars: 2232.29


In [5]:
holdout = dataset.train_test_split(
    test_size=HOLDOUT_SPLIT,
    stratify_by_column="label",
    seed=SEED,
)
valid_test = holdout["test"].train_test_split(
    test_size=TEST_SPLIT / HOLDOUT_SPLIT,
    stratify_by_column="label",
    seed=SEED,
)

dataset = DatasetDict({
    "train": holdout["train"],
    "validation": valid_test["train"],
    "test": valid_test["test"],
})

for split in ["train", "validation", "test"]:
    labels = dataset[split]["label"]
    spam = labels.count(1)
    ham = labels.count(0)
    print(f"{split}: {len(dataset[split])} rows — spam: {spam}, ham: {ham}")


train: 77470 rows — spam: 38735, ham: 38735
validation: 487 rows — spam: 244, ham: 243
test: 3249 rows — spam: 1624, ham: 1625


## Tokenizer & Prompt Format

Qwen3 is trained as a causal LM. We format spam detection as instruction tuning:
the prompt contains the email and the target completion is a single label.


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"pad_token: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")
print(f"eos_token: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"<|im_start|> id: {tokenizer.convert_tokens_to_ids('<|im_start|>')}")
print(f"<|im_end|> id: {tokenizer.convert_tokens_to_ids('<|im_end|>')}")
print(f"<think> id: {tokenizer.convert_tokens_to_ids('<think>')}")
print(f"</think> id: {tokenizer.convert_tokens_to_ids('</think>')}")

for label_text in [POSITIVE_LABEL_TEXT, NEGATIVE_LABEL_TEXT, "valid", "yes", "no"]:
    encoded = tokenizer(label_text, add_special_tokens=False).input_ids
    print(f"{label_text!r}: token_ids={encoded}, token_count={len(encoded)}")


pad_token: '<|endoftext|>' (id=151643)
eos_token: '<|im_end|>' (id=151645)
<|im_start|> id: 151644
<|im_end|> id: 151645
<think> id: 151667
</think> id: 151668
'spam': token_ids=[75545], token_count=1
'ham': token_ids=[5604], token_count=1
'valid': token_ids=[1891], token_count=1
'yes': token_ids=[9693], token_count=1
'no': token_ids=[2152], token_count=1


In [7]:
def build_email_text(subject: str | None, body: str | None) -> str:
    subject = (subject or "").strip()
    body = (body or "").strip()
    parts = []
    if subject:
        parts.append(f"Subject: {subject}")
    if body:
        parts.append(body)
    return "\n\n".join(parts).strip()


def build_user_prompt(email_text: str) -> str:
    return (
        "You are an email spam classifier.\n"
        f"Classify the following email as {POSITIVE_LABEL_TEXT} or {NEGATIVE_LABEL_TEXT}.\n"
        f"Return exactly one lowercase word: {POSITIVE_LABEL_TEXT} or {NEGATIVE_LABEL_TEXT}.\n\n"
        "Email:\n"
        f"{email_text}"
    )


def build_training_example(sample):
    email_text = build_email_text(sample["subject"], sample["body"])
    label_text = POSITIVE_LABEL_TEXT if int(sample["label"]) == 1 else NEGATIVE_LABEL_TEXT
    prompt_messages = [{"role": "user", "content": build_user_prompt(email_text)}]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    return {
        "text": email_text,
        "user_prompt": build_user_prompt(email_text),
        "prompt": prompt_text,
        "completion": f"{label_text}<|im_end|>",
        "label_text": label_text,
    }


dataset = dataset.map(build_training_example, desc="Formatting prompts")
dataset = dataset.filter(lambda sample: bool(sample["text"].strip()), desc="Filtering empty texts")

preview = dataset["train"][0]
print(preview["prompt"])
print("--- completion ---")
print(preview["completion"])


<|im_start|>user
You are an email spam classifier.
Classify the following email as spam or ham.
Return exactly one lowercase word: spam or ham.

Email:
Subject: Corel Draw

OEM means Original Equipment Manufacturer. So OEM is synonym for lowest price.
OEM software means no CD/DVD, no packing case, no booklets and no overhead cost!

Buy directly from the manufacturer, pay for software ONLY and save 75-90%!

Check discounts and special offers! Find software for home and office!
           TOP ITEMS

Windows XP Pro w/SP2          $49
MS Office Enterprise 2007     $79
Adobe Acrobat 8 Pro           $79
Microsoft Windows Vista Ult   $79
Macromedia Studio 8           $99
Adobe Premiere 2.0            $59
Corel Grafix Suite X3         $59
Adobe Illustrator CS2         $59
Macromedia Flash Prof 8       $49
Adobe Photoshop CS2 V9.0      $69
Macromedia Studio 8           $99
Autodesk Autocad 2007        $129
Adobe Creative Suite 2       $149
http://imp.zhopoemka.com/?5F937505ACC8ECD05A424DEDC3234

## Model & LoRA


In [8]:
if CUDA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        dtype=torch.bfloat16,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float32,
    )

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [9]:
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, peft_config)
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
model.print_trainable_parameters()


trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650


## Training


In [10]:
if CUDA:
    print("✅ CUDA detected: using paged_adamw_8bit + bf16.")
    target_optim = "paged_adamw_8bit"
else:
    print("⚠️ CUDA not available: using adamw_torch.")
    target_optim = "adamw_torch"

effective_batch_size = TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
train_batches_per_epoch = int(np.ceil(len(dataset["train"]) / TRAIN_BATCH_SIZE))
optimizer_steps_per_epoch = int(np.ceil(train_batches_per_epoch / GRADIENT_ACCUMULATION_STEPS))
eval_steps = max(1, optimizer_steps_per_epoch // 4)
print(f"Max sequence length: {MAX_SEQ_LENGTH}")
print(f"Effective batch size: {effective_batch_size}")
print(f"Optimizer steps per epoch: {optimizer_steps_per_epoch}")
print(f"Total optimizer steps: {optimizer_steps_per_epoch * NUM_TRAIN_EPOCHS}")
print(f"Evaluation/save cadence: every {eval_steps} optimizer steps")

training_args = SFTConfig(
    report_to=["tensorboard"],
    run_name=AIM_EXPERIMENT_NAME,
    output_dir="./results/qwen3_0.6b_spam_regular_mps",
    logging_dir=f"./runs/{AIM_EXPERIMENT_NAME}",
    logging_strategy="steps",
    logging_steps=10,
    logging_first_step=True,
    save_total_limit=3,
    seed=SEED,
    data_seed=SEED,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    lr_scheduler_type="cosine",

    optim=target_optim,
    max_steps=-1,

    bf16=CUDA,
    fp16=False,
    gradient_checkpointing=False,

    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=600,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,
    packing=False,
    eos_token="<|im_end|>",

    dataloader_pin_memory=True,
    dataloader_num_workers=4,
    remove_unused_columns=False,
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ CUDA detected: using paged_adamw_8bit + bf16.
Max sequence length: 512
Effective batch size: 24
Optimizer steps per epoch: 3228
Total optimizer steps: 3228
Evaluation/save cadence: every 807 optimizer steps


In [11]:
run_config = {
    "model_id": MODEL_ID,
    "seed": SEED,
    "max_seq_length": MAX_SEQ_LENGTH,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "effective_batch_size": effective_batch_size,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "max_grad_norm": MAX_GRAD_NORM,
    "optim": training_args.optim,
    "device": DEVICE,
    "bf16": bool(training_args.bf16),
    "fp16": bool(training_args.fp16),
    "gradient_checkpointing": bool(training_args.gradient_checkpointing),
    "eval_steps": training_args.eval_steps,
    "save_steps": training_args.save_steps,
    "logging_steps": training_args.logging_steps,
    "tensorboard_log_dir": training_args.logging_dir,
    "output_dir": training_args.output_dir,
    "positive_label_text": POSITIVE_LABEL_TEXT,
    "negative_label_text": NEGATIVE_LABEL_TEXT,
}

dataset_metadata = {
    "path": DATASET_PATH,
    "sha256": DATASET_SHA256,
    "rows": sum(len(dataset[split]) for split in dataset),
    "spam": dataset_label_counts["spam"],
    "ham": dataset_label_counts["ham"],
    "sources": dataset_source_counts,
    "avg_subject_chars": dataset_text_stats["avg_subject_chars"],
    "avg_body_chars": dataset_text_stats["avg_body_chars"],
    "train_rows": len(dataset["train"]),
    "validation_rows": len(dataset["validation"]),
    "test_rows": len(dataset["test"]),
}

lora_metadata = {
    "r": LORA_R,
    "alpha": LORA_ALPHA,
    "dropout": LORA_DROPOUT,
    "target_modules": [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    "task_type": "CAUSAL_LM",
}

aim_callback, notebook_aim_callback = create_aim_callbacks(
    repo_path=AIM_REPO_PATH,
    experiment_name=AIM_EXPERIMENT_NAME,
    system_tracking_interval=AIM_SYSTEM_TRACKING_INTERVAL,
    run_config=run_config,
    dataset_metadata=dataset_metadata,
    lora_metadata=lora_metadata,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    callbacks=[aim_callback, notebook_aim_callback],
)


/home/ubuntu/masters-thesis/.venv/lib/python3.12/site-packages/aim/ext/utils.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [12]:
trainer_stats = trainer.train()
model = trainer.model
model.config.use_cache = True
trainer_stats


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss
50,0.110851,0.158417
100,0.085514,0.112804
150,0.054845,0.011934
200,0.125298,0.074978
250,0.034892,0.028959
300,0.019388,0.079848
350,0.055964,0.031268
400,0.000313,0.025436
450,0.001372,0.026069
500,0.044460,0.026646


TrainOutput(global_step=3228, training_loss=0.021670830198017983, metrics={'train_runtime': 3214.3335, 'train_samples_per_second': 24.101, 'train_steps_per_second': 1.004, 'total_flos': 1.072278011904e+17, 'train_loss': 0.021670830198017983})

## Save Adapter


In [13]:
import datetime

timestamp = datetime.datetime.now().strftime("%H%M%d%m%Y")
save_path = f"./results/qwen3_0.6b_spam_regular_saved_weights_{timestamp}"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
aim_callback.experiment["final_saved_model_path"] = save_path
aim_callback.experiment["saved_tokenizer_path"] = save_path

print(f"✅ Model safely saved to disk at: {save_path}")


✅ Model safely saved to disk at: ./results/qwen3_0.6b_spam_regular_saved_weights_222719042026


## Inference

Inference uses Qwen3's hard non-thinking switch (`enable_thinking=False`) so the
model answers directly with a short label instead of reasoning first.


In [14]:
THINK_BLOCK_RE = re.compile(r"<think>.*?</think>", flags=re.DOTALL)


def strip_reasoning_and_special_tokens(text: str) -> str:
    cleaned = THINK_BLOCK_RE.sub(" ", text)
    cleaned = cleaned.replace("<|im_end|>", " ")
    cleaned = cleaned.replace("<|endoftext|>", " ")
    return cleaned.strip()


def parse_generated_label(text: str):
    cleaned = strip_reasoning_and_special_tokens(text).lower()
    first_line = cleaned.splitlines()[0].strip() if cleaned else ""

    if re.search(r"\bspam\b", first_line):
        return POSITIVE_LABEL_TEXT
    if re.search(r"\bham\b", first_line):
        return NEGATIVE_LABEL_TEXT
    if re.search(r"\bvalid\b", first_line):
        return NEGATIVE_LABEL_TEXT
    if re.search(r"\byes\b", first_line):
        return POSITIVE_LABEL_TEXT
    if re.search(r"\bno\b", first_line):
        return NEGATIVE_LABEL_TEXT

    return None


def label_to_id(label_text: str | None) -> int:
    if label_text is None:
        return 0
    return LABEL_TEXT_TO_ID.get(label_text, 0)


def run_mail_classification(email_text: str, max_new_tokens: int = 4):
    prompt_messages = [{"role": "user", "content": build_user_prompt(email_text.strip())}]
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    stop_token_ids = [tokenizer.convert_tokens_to_ids("<|im_end|>")]
    if tokenizer.eos_token_id is not None and tokenizer.eos_token_id not in stop_token_ids:
        stop_token_ids.append(tokenizer.eos_token_id)

    model.eval()
    model_device = next(model.parameters()).device
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model_device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=stop_token_ids,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw_generation = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )
    parsed_label = parse_generated_label(raw_generation)
    return {
        "raw_generation": raw_generation,
        "parsed_label": parsed_label or NEGATIVE_LABEL_TEXT,
        "parse_failed": parsed_label is None,
    }


def run_mail_from_dataset_classification(dataset_split, index: int):
    return run_mail_classification(dataset_split[index]["text"])


## Evaluation


In [15]:
def compute_generation_metrics(predictions, labels):
    predictions = np.asarray(predictions)
    labels = np.asarray(labels)

    tp = int(((predictions == 1) & (labels == 1)).sum())
    fp = int(((predictions == 1) & (labels == 0)).sum())
    fn = int(((predictions == 0) & (labels == 1)).sum())
    tn = int(((predictions == 0) & (labels == 0)).sum())

    accuracy = float((predictions == labels).mean())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    specificity = tn / max(tn + fp, 1)
    balanced_accuracy = (recall + specificity) / 2

    return {
        "accuracy": accuracy,
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "specificity": float(specificity),
        "balanced_accuracy": float(balanced_accuracy),
        "false_positive_count": float(fp),
        "false_negative_count": float(fn),
        "true_positive_count": float(tp),
        "true_negative_count": float(tn),
    }


test_predictions = []
test_true_labels = []
incorrect_samples = []
parse_failures = []

for i, sample in enumerate(dataset["test"]):
    result = run_mail_classification(sample["text"])
    pred_label_text = result["parsed_label"]
    pred_label = label_to_id(pred_label_text)
    actual_label = int(sample["label"])

    test_predictions.append(pred_label)
    test_true_labels.append(actual_label)

    if result["parse_failed"]:
        parse_failures.append({
            "index": i,
            "content": sample["text"],
            "raw_generation": result["raw_generation"],
        })

    if pred_label != actual_label:
        incorrect_samples.append({
            "index": i,
            "content": sample["text"],
            "actual": NEGATIVE_LABEL_TEXT if actual_label == 0 else POSITIVE_LABEL_TEXT,
            "output": pred_label_text,
            "raw_generation": result["raw_generation"],
            "parse_failed": result["parse_failed"],
        })

rounded_metrics = {
    key: round(value, 4)
    for key, value in compute_generation_metrics(test_predictions, test_true_labels).items()
}
print(rounded_metrics)
print(f"Parse failures: {len(parse_failures)} / {len(test_true_labels)}")
print(f"Mistakes: {len(incorrect_samples)} / {len(test_true_labels)}")


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/home/ubuntu/masters-thesis/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{'accuracy': 0.9249, 'precision': 0.9859, 'recall': 0.8621, 'f1': 0.9198, 'specificity': 0.9877, 'balanced_accuracy': 0.9249, 'false_positive_count': 20.0, 'false_negative_count': 224.0, 'true_positive_count': 1400.0, 'true_negative_count': 1605.0}
Parse failures: 794 / 3249
Mistakes: 244 / 3249


In [16]:
for parse_failure in parse_failures:
    print(parse_failure['content'])
    print("CLASS:", parse_failure['raw_generation'])

Subject: Just keep in touch

Please Stace, I've come broken fasten already fat conceded that was aAs harass long happily a matter of fact, just distance the opposite. She's not Stacy shook her head and harmony slit smiled. button I swear Carl,Huh? plastic Dana was thumb zoic caught a collar little off guard.
fled Oh they ancient took me stare alright. bee It's the part about cYour safety is slope more glow cheese recognise important than mine. 
Unbeknownst upheld to Jeff, behavior Aggie had shop volucrine been listening i from trod upset Stacy continued. You know, end as I'm sitting here cup Well, army since both rule of you are brush obviously into thi You alert said that reluctantly you and poor he are imagine going to be study
But you'll only lip fork be excuse able to type precede with one hand.Alright, twist I get the cry idea. taught breathe I take comfort in kno Jeff you leaped plug belief look know that isn't true... journey Jeff laid start back cladistic down. In muddy my dark

In [17]:
print(incorrect_samples)


[{'index': 1, 'content': "Subject: Just keep in touch\n\nPlease Stace, I've come broken fasten already fat conceded that was aAs harass long happily a matter of fact, just distance the opposite. She's not Stacy shook her head and harmony slit smiled. button I swear Carl,Huh? plastic Dana was thumb zoic caught a collar little off guard.\nfled Oh they ancient took me stare alright. bee It's the part about cYour safety is slope more glow cheese recognise important than mine. \nUnbeknownst upheld to Jeff, behavior Aggie had shop volucrine been listening i from trod upset Stacy continued. You know, end as I'm sitting here cup Well, army since both rule of you are brush obviously into thi You alert said that reluctantly you and poor he are imagine going to be study\nBut you'll only lip fork be excuse able to type precede with one hand.Alright, twist I get the cry idea. taught breathe I take comfort in kno Jeff you leaped plug belief look know that isn't true... journey Jeff laid start back c

In [18]:
examples = [
    '''Subject: E-mail details of the client.
Hi Greg,
I have received the following contact info from the apache guys: "dan@apache.com",
I just wanted to check if this information is correct.
Best regards,
John''',
    '''Subject: Free iPhone.
Hi Greg,
You have won a free iPhone. Press the following link to receive your reward:
"http://free-iphone.com"''',
    '''Subject: Wojciech, your virtual card is ready for use.
Your Revolut virtual card is ready for you. Head over to the Cards tab in the app
to access your new card details and start making payments online.''',
    '''Subject: Obsługa języka polskiego.
Kup najnowszy ajfon za prawie darmo niskie ceny loteria, jesteś tysięcznym użytkownikiem!
Pozdrawiam,
Wojciech''',
    '''How are you doing? Can we meet later this afternoon?
I just wanted to check if this information is correct.''',
]

for example in examples:
    result = run_mail_classification(example)
    print("=" * 80)
    print(example)
    print(f"raw_generation={result['raw_generation']!r}")
    print(f"parsed_label={result['parsed_label']!r}")
    print(f"parse_failed={result['parse_failed']}")


Subject: E-mail details of the client.
Hi Greg,
I have received the following contact info from the apache guys: "dan@apache.com",
I just wanted to check if this information is correct.
Best regards,
John
raw_generation='ham<|im_end|>'
parsed_label='ham'
parse_failed=False
Subject: Free iPhone.
Hi Greg,
You have won a free iPhone. Press the following link to receive your reward:
"http://free-iphone.com"
raw_generation='spam<|im_end|>'
parsed_label='spam'
parse_failed=False
Subject: Wojciech, your virtual card is ready for use.
Your Revolut virtual card is ready for you. Head over to the Cards tab in the app
to access your new card details and start making payments online.
raw_generation='spam<|im_end|>'
parsed_label='spam'
parse_failed=False
Subject: Obsługa języka polskiego.
Kup najnowszy ajfon za prawie darmo niskie ceny loteria, jesteś tysięcznym użytkownikiem!
Pozdrawiam,
Wojciech
raw_generation='spam<|im_end|>'
parsed_label='spam'
parse_failed=False
How are you doing? Can we meet 